# 01 - Loading and Cleaning

Read an uploaded file (CSV, TSV, Excel, JSON, Parquet), find the real
header row and clean it automatically before anything reaches SQLite or
the model.

- **Applied automatically:** header detection, column names, empty rows
  and columns, whitespace, null placeholders, numbers / booleans / dates
  stored as text, category spelling variants, exact duplicate rows.
- **Flagged, never changed:** missing values, outliers, ambiguous dates,
  similar but different categories, partly numeric columns, identifiers
  with leading zeros, constant columns.

Every table is returned twice: `cleaned` and `raw` (original values,
cleaned column names only), plus a cleaning report.

Becomes `src/helpers/file_loader.py` and `src/helpers/data_cleaner.py`.

In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import csv
import difflib
import io
import json
import logging
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd


# Project root: works from the notebook folder or the project root.
BASE_DIR = (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)
INPUT_DIR = BASE_DIR / "data" / "input_data"
OUTPUT_DIR = BASE_DIR / "data" / "output_data" / "cleaned_data"

SUPPORTED_EXTENSIONS = {
    ".csv", ".tsv", ".xlsx", ".xlsm", ".json", ".jsonl", ".parquet"
}
MAX_FILE_SIZE_MB = 50

# Text decoding attempts, in order (latin-1 never fails).
TEXT_ENCODINGS = ["utf-8-sig", "cp1252", "latin-1"]
CSV_DELIMITERS = [",", ";", "\t", "|"]

# Header detection
HEADER_SCAN_ROWS = 20
HEADER_MIN_FILLED_RATIO = 0.8

# A column is converted when at least this share of its non-null
# values parse; between PARTLY_TYPED_RATIO and this it is only flagged.
# A single value that fails is also tolerated when the rest is at least
# SINGLE_FAILURE_MIN_RATIO (one stray "unknown" in a small table).
TYPE_MATCH_THRESHOLD = 0.95
SINGLE_FAILURE_MIN_RATIO = 0.9
PARTLY_TYPED_RATIO = 0.5

# Category handling
CATEGORY_MAX_UNIQUE_RATIO = 0.5
SIMILAR_CATEGORY_RATIO = 0.8
SIMILAR_CATEGORY_MAX_VALUES = 100

# Outliers: values beyond Q1 - k*IQR or Q3 + k*IQR, but only when they
# are rare (many "extreme" values just means a skewed column), plus the
# top or bottom values when they jump by GAP times from the next value.
OUTLIER_IQR_FACTOR = 3.0
OUTLIER_MAX_SHARE = 0.05
OUTLIER_GAP_FACTOR = 10

NULL_PLACEHOLDERS = {
    "", "na", "n/a", "n.a.", "nan", "null", "none", "nil", "-", "--",
    "?", "missing", "#n/a", "not available", "undefined",
}
TRUE_VALUES = {"yes", "y", "true", "t"}
FALSE_VALUES = {"no", "n", "false", "f"}

CURRENCY_SYMBOLS = "$€£₹¥"
CURRENCY_PREFIX_PATTERN = re.compile(
    r"^(rs\.?|inr|usd|eur|gbp)\s*", re.IGNORECASE
)
NUMBER_PATTERN = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$")
LEADING_ZERO_PATTERN = re.compile(r"^[+-]?0\d")

# Formats tried for dates stored as text.
DATE_FORMATS = [
    "%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y", "%m-%d-%Y", "%d/%m/%Y",
    "%m/%d/%Y", "%d.%m.%Y", "%d/%m/%y", "%m/%d/%y", "%d-%b-%Y",
    "%d %b %Y", "%b %d, %Y", "%d %B %Y", "%B %d, %Y",
    "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M",
    "%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M",
]

# Column names that would need quoting in SQL get a suffix.
SQL_RESERVED_WORDS = {
    "all", "and", "any", "as", "asc", "between", "by", "case", "cast",
    "check", "column", "create", "default", "delete", "desc", "distinct",
    "drop", "else", "end", "exists", "from", "group", "having", "in",
    "index", "insert", "into", "is", "join", "key", "like", "limit",
    "not", "null", "offset", "on", "or", "order", "primary",
    "references", "select", "set", "table", "then", "to", "union",
    "update", "values", "when", "where",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("01_loading_cleaning")

def sample_files():
    """
    Sample data files directly in INPUT_DIR. Sub-folders (eval/) and the
    question sets (*_questions.json) are not data.
    """

    return [path for path in sorted(INPUT_DIR.iterdir())
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
            and not path.name.endswith("_questions.json")]


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

Input folder : G:\GEN AI CE2\data\input_data
Output folder: G:\GEN AI CE2\data\output_data\cleaned_data


## 2. Reading files

Every reader returns a list of tables. CSV and Excel tables come back as
a **grid** (no header yet, every cell as text or `None`) so the header
row can be detected; JSON and Parquet already have column names.

In [2]:
# ============================================================
# 2. READING FILES
# ============================================================

def validate_file(file_path):
    """Check the file exists, has a supported type and is not too big."""

    # Step 1: the file must exist.
    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {file_path}")

    # Step 2: supported extension only.
    extension = file_path.suffix.lower()

    if extension not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file type {extension!r}. Supported: "
            f"{', '.join(sorted(SUPPORTED_EXTENSIONS))}"
        )

    # Step 3: size limit, and not empty.
    size_mb = file_path.stat().st_size / (1024 * 1024)

    if size_mb == 0:
        raise ValueError(f"File is empty: {file_path.name}")

    if size_mb > MAX_FILE_SIZE_MB:
        raise ValueError(
            f"File is {size_mb:.1f} MB; the limit is {MAX_FILE_SIZE_MB} MB."
        )

    return file_path


def cell_to_text(value):
    """
    Turn one cell into stripped text, or None when it is empty.

    Excel dates become ISO text and whole floats lose their ".0", so
    every source is cleaned by the same text rules.
    """

    if value is None:
        return None

    if isinstance(value, float) and pd.isna(value):
        return None

    if isinstance(value, (pd.Timestamp, datetime)):
        if (value.hour, value.minute, value.second) == (0, 0, 0):
            return value.strftime("%Y-%m-%d")
        return value.strftime("%Y-%m-%d %H:%M:%S")

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)

    text = str(value).strip()

    return text if text else None


def read_text(file_path):
    """Decode a text file with the first encoding that works."""

    raw_bytes = file_path.read_bytes()

    for encoding in TEXT_ENCODINGS:
        try:
            return raw_bytes.decode(encoding), encoding

        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not decode {file_path.name} as text.")


def detect_delimiter(text, file_path):
    """
    Pick the delimiter that splits the most lines into the same number
    of fields. Title lines above the header do not confuse it.
    """

    if file_path.suffix.lower() == ".tsv":
        return "\t"

    lines = [line for line in text.splitlines() if line.strip()][:50]
    best_delimiter, best_score = ",", 0

    for delimiter in CSV_DELIMITERS:
        counts = [line.count(delimiter) for line in lines]
        non_zero = [count for count in counts if count > 0]

        if not non_zero:
            continue

        # Lines sharing the most common field count.
        common_count = max(set(non_zero), key=non_zero.count)
        score = non_zero.count(common_count)

        if score > best_score:
            best_delimiter, best_score = delimiter, score

    return best_delimiter


def rows_to_grid(rows):
    """Pad ragged rows into a rectangular DataFrame of text cells."""

    width = max((len(row) for row in rows), default=0)
    padded = [
        [cell_to_text(cell) for cell in row] + [None] * (width - len(row))
        for row in rows
    ]

    return pd.DataFrame(padded, dtype=object)


def read_csv_file(file_path):
    """CSV / TSV -> one grid table."""

    text, encoding = read_text(file_path)
    delimiter = detect_delimiter(text, file_path)
    rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))

    logger.info(
        "Read %s | encoding=%s | delimiter=%r | lines=%d",
        file_path.name, encoding, delimiter, len(rows),
    )

    return [{
        "sheet": None,
        "grid": rows_to_grid(rows),
        "header_known": False,
        "source_info": {"encoding": encoding, "delimiter": delimiter},
    }]


def read_excel_file(file_path):
    """Excel -> one grid table per non-empty sheet."""

    sheets = pd.read_excel(
        file_path, sheet_name=None, header=None, dtype=object,
        engine="openpyxl",
    )
    tables = []

    for sheet_name, sheet in sheets.items():
        # rows_to_grid keeps empty cells as None (not NaN).
        grid = rows_to_grid(sheet.astype(object).values.tolist())

        if grid.isna().all().all():
            logger.info("Skipping empty sheet %r", sheet_name)
            continue

        tables.append({
            "sheet": str(sheet_name),
            "grid": grid,
            "header_known": False,
            "source_info": {"sheet": str(sheet_name)},
        })

    logger.info("Read %s | sheets=%d", file_path.name, len(tables))

    return tables


def read_json_file(file_path):
    """
    JSON / JSON Lines -> one table.

    Accepts a list of records, a dict holding one list of records
    (e.g. {"customers": [...]}) or one record per line. Nested objects
    are flattened into columns such as address_city.
    """

    text, _ = read_text(file_path)

    # Step 1: whole document first, then JSON Lines.
    try:
        data = json.loads(text)

    except json.JSONDecodeError:
        try:
            data = [json.loads(line) for line in text.splitlines()
                    if line.strip()]

        except json.JSONDecodeError as exc:
            raise ValueError(f"{file_path.name} is not valid JSON: {exc}")

    # Step 2: find the list of records.
    if isinstance(data, dict):
        record_lists = [
            value for value in data.values()
            if isinstance(value, list) and value
            and all(isinstance(item, dict) for item in value)
        ]
        data = record_lists[0] if len(record_lists) == 1 else [data]

    if not isinstance(data, list) or not data:
        raise ValueError(f"{file_path.name} has no records to load.")

    # Step 3: flatten nested objects.
    frame = pd.json_normalize(data, sep="_")

    logger.info("Read %s | records=%d", file_path.name, len(frame))

    return [{
        "sheet": None,
        "grid": frame_to_grid(frame),
        "header_known": True,
        "source_info": {},
    }]


def read_parquet_file(file_path):
    """Parquet -> one table (types are re-inferred like any other file)."""

    frame = pd.read_parquet(file_path)

    logger.info("Read %s | rows=%d", file_path.name, len(frame))

    return [{
        "sheet": None,
        "grid": frame_to_grid(frame),
        "header_known": True,
        "source_info": {},
    }]


def frame_to_grid(frame):
    """Put a frame's column names back as row 0 of a text grid."""

    rows = [list(frame.columns)] + frame.astype(object).values.tolist()

    return rows_to_grid(rows)


def read_file(file_path):
    """Validate a file and read it into a list of grid tables."""

    file_path = validate_file(file_path)
    extension = file_path.suffix.lower()

    if extension in {".csv", ".tsv"}:
        return read_csv_file(file_path)

    if extension in {".xlsx", ".xlsm"}:
        return read_excel_file(file_path)

    if extension in {".json", ".jsonl"}:
        return read_json_file(file_path)

    return read_parquet_file(file_path)


for sample in sample_files():
    for table in read_file(sample):
        print(f"{sample.name:<22} sheet={table['sheet']!s:<10} "
              f"grid={table['grid'].shape}")

2026-11-14 19:25:50,219 | INFO | Read customers.json | records=8


customers.json         sheet=None       grid=(9, 6)


2026-11-14 19:25:50,506 | INFO | Read messy_employees.xlsx | sheets=2


2026-11-14 19:25:50,508 | INFO | Read messy_sales.csv | encoding=utf-8-sig | delimiter=',' | lines=39


2026-11-14 19:25:50,540 | INFO | Read products.parquet | rows=5


messy_employees.xlsx   sheet=Employees  grid=(23, 7)
messy_employees.xlsx   sheet=Targets    grid=(5, 3)
messy_sales.csv        sheet=None       grid=(39, 15)
products.parquet       sheet=None       grid=(6, 3)


## 3. Finding the header row

The header is the first row (within the first 20) that is nearly as full
as the widest row and made mostly of text. Title rows such as
"Quarterly Sales Report" fill one cell, so they are skipped.

In [3]:
# ============================================================
# 3. FINDING THE HEADER ROW
# ============================================================

def parse_number(text):
    """
    Parse a number stored as text.

    Handles currency symbols, thousands separators, percent signs and
    accounting negatives "(1,200)". Internal spaces are refused so phone
    numbers stay text.

    Returns:
        (float, unit) or None. unit is "currency", "percent" or None.
    """

    text = str(text).strip()
    unit = None
    negative = False

    # Step 1: accounting negative "(1,200)".
    if text.startswith("(") and text.endswith(")"):
        negative = True
        text = text[1:-1].strip()

    # Step 2: percent sign.
    if text.endswith("%"):
        unit = "percent"
        text = text[:-1].strip()

    # Step 3: currency symbols and codes.
    for symbol in CURRENCY_SYMBOLS:
        if symbol in text:
            unit = unit or "currency"
            text = text.replace(symbol, "")

    if CURRENCY_PREFIX_PATTERN.match(text):
        unit = unit or "currency"
        text = CURRENCY_PREFIX_PATTERN.sub("", text)

    # Step 4: thousands separators, then a plain number must remain.
    text = text.strip().replace(",", "")

    if not NUMBER_PATTERN.match(text):
        return None

    number = float(text)

    return (-number if negative else number), unit


def detect_header_row(grid):
    """Return the index of the header row in a text grid."""

    filled_counts = grid.notna().sum(axis=1)
    max_filled = int(filled_counts.max())
    min_filled = max(min(2, max_filled),
                     int(HEADER_MIN_FILLED_RATIO * max_filled))

    for index in grid.index[:HEADER_SCAN_ROWS]:
        values = [value for value in grid.loc[index] if pd.notna(value)]

        if len(values) < min_filled:
            continue

        text_values = [value for value in values
                       if parse_number(value) is None]

        if len(text_values) >= HEADER_MIN_FILLED_RATIO * len(values):
            return index

    # Fallback: the first fullest row.
    return int(filled_counts.idxmax())


def split_header(table):
    """Return (header names, body grid, rows dropped above the header)."""

    grid = table["grid"]
    header_row = 0 if table["header_known"] else detect_header_row(grid)

    headers = list(grid.iloc[header_row])
    body = grid.iloc[header_row + 1:].reset_index(drop=True)
    body.columns = range(len(headers))

    return headers, body, header_row


for sample in sample_files():
    for table in read_file(sample):
        headers, body, dropped = split_header(table)
        print(f"{sample.name:<22} header row={dropped} "
              f"rows above dropped={dropped} -> {headers}")

2026-11-14 19:25:50,554 | INFO | Read customers.json | records=8


2026-11-14 19:25:50,566 | INFO | Read messy_employees.xlsx | sheets=2


2026-11-14 19:25:50,568 | INFO | Read messy_sales.csv | encoding=utf-8-sig | delimiter=',' | lines=39


2026-11-14 19:25:50,572 | INFO | Read products.parquet | rows=5


customers.json         header row=0 rows above dropped=0 -> ['customer_id', 'name', 'signup', 'tags', 'address_city', 'address_pin']
messy_employees.xlsx   header row=2 rows above dropped=2 -> ['Emp ID', 'Full Name', 'Department', 'Joining Date', 'Salary (INR)', 'Last Review', 'Active']
messy_employees.xlsx   header row=0 rows above dropped=0 -> ['Department', 'Q1 Target', 'Q2 Target']
messy_sales.csv        header row=3 rows above dropped=3 -> ['Order ID', 'Order Date', 'Customer Name', 'City', 'Region', 'Product', 'Units Sold', 'Unit Price ($)', 'Discount %', 'Total Sales ($)', 'Paid?', 'Sales Rep', 'Sales Rep', 'Notes', None]
products.parquet       header row=0 rows above dropped=0 -> ['product', 'category', 'list_price']


## 4. Column names

`Total Sales ($)` -> `total_sales`, `Discount %` -> `discount_pct`,
empty names -> `column_N`, duplicates -> `_2`, SQL keywords -> `_value`.

In [4]:
# ============================================================
# 4. COLUMN NAMES
# ============================================================

def clean_column_name(name, position):
    """Turn one header into a lowercase snake_case SQL-safe name."""

    text = "" if name is None or pd.isna(name) else str(name)

    # Step 1: drop units in brackets, keep meaning of a few symbols.
    text = re.sub(r"\(.*?\)|\[.*?\]", " ", text)
    text = (text.replace("%", " pct ").replace("#", " number ")
                .replace("&", " and "))

    # Step 2: accents -> ASCII, then snake_case.
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9]+", "_", text.casefold()).strip("_")

    # Step 3: names SQL cannot use bare.
    if not text:
        text = f"column_{position + 1}"

    if text[0].isdigit():
        text = f"col_{text}"

    if text in SQL_RESERVED_WORDS:
        text = f"{text}_value"

    return text


def clean_column_names(headers):
    """
    Clean every header and make the names unique.

    Returns:
        (list of new names, dict of original -> new for changed names)
    """

    names = []
    renamed = {}
    used = set()

    for position, header in enumerate(headers):
        base = clean_column_name(header, position)
        name = base
        suffix = 2

        while name in used:
            name = f"{base}_{suffix}"
            suffix += 1

        used.add(name)
        names.append(name)

        original = "" if header is None or pd.isna(header) else str(header)

        if original != name:
            renamed[original if original else f"<empty #{position + 1}>"] = name

    return names, renamed


for sample in sample_files():
    for table in read_file(sample):
        headers, _, _ = split_header(table)
        names, _ = clean_column_names(headers)
        print(f"{sample.name:<22} {names}")

2026-11-14 19:25:50,584 | INFO | Read customers.json | records=8


2026-11-14 19:25:50,594 | INFO | Read messy_employees.xlsx | sheets=2


2026-11-14 19:25:50,596 | INFO | Read messy_sales.csv | encoding=utf-8-sig | delimiter=',' | lines=39


2026-11-14 19:25:50,600 | INFO | Read products.parquet | rows=5


customers.json         ['customer_id', 'name', 'signup', 'tags', 'address_city', 'address_pin']
messy_employees.xlsx   ['emp_id', 'full_name', 'department', 'joining_date', 'salary', 'last_review', 'active']
messy_employees.xlsx   ['department', 'q1_target', 'q2_target']
messy_sales.csv        ['order_id', 'order_date', 'customer_name', 'city', 'region', 'product', 'units_sold', 'unit_price', 'discount_pct', 'total_sales', 'paid', 'sales_rep', 'sales_rep_2', 'notes', 'column_15']
products.parquet       ['product', 'category', 'list_price']


## 5. Cleaning values

Whitespace, null placeholders, empty rows and columns, exact duplicates.
Each step returns the frame and records what it did in the report.

In [5]:
# ============================================================
# 5. CLEANING VALUES
# ============================================================

def new_report(table_name, source_file):
    """Empty cleaning report for one table."""

    return {
        "table_name": table_name,
        "source_file": source_file,
        "source_info": {},
        "header_row": 0,
        "rows_before": 0,
        "rows_after": 0,
        "columns_before": 0,
        "columns_after": 0,
        "renamed_columns": {},
        "column_types": {},
        "column_units": {},
        "actions": [],
        "flags": [],
    }


def add_action(report, column, action, detail, count=None):
    """Record a change that was applied."""

    report["actions"].append({
        "column": column, "action": action, "detail": detail,
        "count": count,
    })


def add_flag(report, column, issue, detail):
    """Record a problem that was left unchanged."""

    report["flags"].append({"column": column, "issue": issue,
                            "detail": detail})


def trim_whitespace(frame, report):
    """Strip and collapse whitespace inside every text cell."""

    changed = 0

    for column in frame.columns:
        original = frame[column]
        trimmed = original.map(
            lambda value: re.sub(r"\s+", " ", value).strip()
            if isinstance(value, str) else value
        )
        changed += int((original.fillna("") != trimmed.fillna("")).sum())
        frame[column] = trimmed

    if changed:
        add_action(report, None, "trimmed_whitespace",
                   "Removed extra spaces inside text values.", changed)

    return frame


def replace_null_placeholders(frame, report):
    """N/A, -, null, ? ... become real nulls."""

    for column in frame.columns:
        is_placeholder = frame[column].map(
            lambda value: isinstance(value, str)
            and value.casefold() in NULL_PLACEHOLDERS
        )
        count = int(is_placeholder.sum())

        if count:
            found = sorted(set(frame.loc[is_placeholder, column]))
            frame.loc[is_placeholder, column] = None
            add_action(report, column, "null_placeholders",
                       f"Treated {found} as missing.", count)

    return frame


def drop_empty_rows_and_columns(frame, report):
    """Remove rows and columns with no values at all."""

    empty_columns = [column for column in frame.columns
                     if frame[column].isna().all()]

    if empty_columns:
        frame = frame.drop(columns=empty_columns)
        add_action(report, None, "dropped_empty_columns",
                   f"Removed empty columns {empty_columns}.",
                   len(empty_columns))

    empty_rows = frame.isna().all(axis=1)

    if empty_rows.any():
        frame = frame.loc[~empty_rows].reset_index(drop=True)
        add_action(report, None, "dropped_empty_rows",
                   "Removed rows with no values.", int(empty_rows.sum()))

    return frame


def drop_duplicate_rows(frame, report):
    """Remove exact duplicate rows, keeping the first."""

    duplicates = frame.duplicated(keep="first")

    if duplicates.any():
        frame = frame.loc[~duplicates].reset_index(drop=True)
        add_action(report, None, "dropped_duplicate_rows",
                   "Removed exact duplicate rows.", int(duplicates.sum()))

    return frame

## 6. Detecting column types

Order: boolean -> number -> date -> text. A type is applied only when at
least 95 % of the non-null values parse (or all but one value, when that
is still 90 %); the values that fail become null and are listed in the report. Day-first and month-first date
formats that both fit equally well are **ambiguous**, so the column is
left as text and flagged.

In [6]:
# ============================================================
# 6. DETECTING COLUMN TYPES
# ============================================================

def type_matches(matched, total):
    """
    True when enough values parse: at least TYPE_MATCH_THRESHOLD, or all
    but one when that one leaves at least SINGLE_FAILURE_MIN_RATIO.
    """

    if total == 0:
        return False

    rate = matched / total

    return (rate >= TYPE_MATCH_THRESHOLD
            or (total - matched == 1 and rate >= SINGLE_FAILURE_MIN_RATIO))


def try_boolean(values):
    """Yes/No, Y/N, True/False -> booleans."""

    lowered = values.str.casefold()
    known = lowered.isin(TRUE_VALUES | FALSE_VALUES)

    if not type_matches(int(known.sum()), len(known)):
        return None

    converted = lowered.map(
        lambda value: True if value in TRUE_VALUES
        else False if value in FALSE_VALUES else None
    )

    return {"kind": "boolean", "values": converted,
            "failed": values[~known], "unit": None}


def try_number(values):
    """
    Numbers stored as text -> integers or floats.

    Returns a result dict, or a dict with kind "partly_numeric" /
    "leading_zero" when the column is only flagged.
    """

    # Step 1: identifiers such as 00123 must keep their zeros.
    if values.str.match(LEADING_ZERO_PATTERN).any():
        if type_matches(int(values.map(parse_number).notna().sum()), len(values)):
            return {"kind": "leading_zero"}
        return None

    # Step 2: parse every value.
    parsed = values.map(parse_number)
    ok = parsed.notna()
    rate = ok.mean()

    if not type_matches(int(ok.sum()), len(ok)):
        if rate >= PARTLY_TYPED_RATIO:
            return {"kind": "partly_numeric", "rate": rate,
                    "failed": values[~ok]}
        return None

    # Step 3: integers when every number is whole.
    numbers = parsed[ok].map(lambda result: result[0])
    units = {result[1] for result in parsed[ok] if result[1]}
    is_whole = numbers.map(lambda number: number.is_integer()).all()

    converted = pd.Series(pd.NA, index=values.index, dtype="Float64")
    converted[ok] = numbers

    if is_whole and numbers.abs().max() < 2 ** 53:
        converted = converted.astype("Int64")

    return {
        "kind": "integer" if is_whole else "float",
        "values": converted,
        "failed": values[~ok],
        "unit": "/".join(sorted(units)) if units else None,
    }


def swap_day_month(date_format):
    """%d/%m/%Y <-> %m/%d/%Y"""

    return (date_format.replace("%d", "@").replace("%m", "%d")
                       .replace("@", "%m"))


def try_date(values):
    """
    Dates stored as text -> datetimes.

    Every known format is tried; the best one wins. When the day-first
    and month-first versions fit equally well the column is ambiguous.
    """

    results = []

    for date_format in DATE_FORMATS:
        parsed = pd.to_datetime(values, format=date_format, errors="coerce")
        rate = parsed.notna().mean()

        if type_matches(int(parsed.notna().sum()), len(parsed)):
            results.append((rate, date_format, parsed))

    if not results:
        return None

    results.sort(key=lambda result: result[0], reverse=True)
    best_rate, best_format, best_parsed = results[0]

    # Ambiguity: the swapped format fits just as well.
    swapped = swap_day_month(best_format)

    if swapped != best_format and any(
        date_format == swapped and rate == best_rate
        for rate, date_format, _ in results
    ):
        return {"kind": "ambiguous_date",
                "formats": [best_format, swapped]}

    has_time = (
        best_parsed.dropna().dt.normalize() != best_parsed.dropna()
    ).any()

    return {
        "kind": "datetime" if has_time else "date",
        "values": best_parsed,
        "failed": values[best_parsed.isna()],
        "unit": None,
        "format": best_format,
    }


def convert_column(frame, column, report):
    """Infer one column's type, convert it and record the outcome."""

    values = frame[column].dropna().astype(str)

    if values.empty:
        report["column_types"][column] = "text"
        return frame

    for attempt in (try_boolean, try_number, try_date):
        result = attempt(values)

        if result is None:
            continue

        kind = result["kind"]

        # Flag-only outcomes: keep the text, explain why.
        if kind == "leading_zero":
            add_flag(report, column, "identifier_kept_as_text",
                     "Values have leading zeros (e.g. "
                     f"{values.iloc[0]!r}), so they stay text.")
            break

        if kind == "partly_numeric":
            add_flag(report, column, "partly_numeric",
                     f"{result['rate']:.0%} of values are numbers; kept as "
                     f"text. Not numbers: "
                     f"{sorted(set(result['failed']))[:5]}")
            break

        if kind == "ambiguous_date":
            add_flag(report, column, "ambiguous_dates",
                     "Dates fit both day-first and month-first "
                     f"({' / '.join(result['formats'])}); kept as text.")
            break

        # A real conversion.
        # Rows that were null before stay null (reindex fills them).
        converted = result["values"].reindex(frame.index)

        if kind == "boolean":
            converted = converted.astype("boolean")

        frame[column] = converted

        detail = f"Converted text to {kind}"

        if result.get("format"):
            detail += f" (format {result['format']})"

        if result["unit"]:
            detail += f"; removed {result['unit']} symbols"
            report["column_units"][column] = result["unit"]

        failed = sorted(set(result["failed"]))

        if failed:
            detail += f"; values that did not parse became null: {failed[:5]}"

        add_action(report, column, f"converted_to_{kind}", detail + ".",
                   len(values))
        report["column_types"][column] = kind
        return frame

    report["column_types"][column] = "text"

    return frame

## 7. Categories and flags

Case and spacing variants of the same category are merged into the most
common spelling. Everything else only becomes a flag.

In [7]:
# ============================================================
# 7. CATEGORIES AND FLAGS
# ============================================================

def is_category_column(series):
    """Low-cardinality text column (city, product, status ...)."""

    values = series.dropna()

    return (not values.empty
            and values.nunique() <= CATEGORY_MAX_UNIQUE_RATIO * len(values))


def merge_category_variants(frame, report):
    """'delhi', 'DELHI' -> the most common spelling, 'Delhi'."""

    for column, kind in report["column_types"].items():
        if kind != "text" or not is_category_column(frame[column]):
            continue

        counts = frame[column].value_counts()
        groups = {}

        for value in counts.index:
            groups.setdefault(value.casefold(), []).append(value)

        mapping = {}

        for variants in groups.values():
            if len(variants) > 1:
                # value_counts is sorted by frequency. Prefer the most
                # common normally cased spelling ("Beverages" over
                # "BEVERAGES"), then any not all lowercase ("HR" over
                # "hr"), then simply the most common.
                proper = [variant for variant in variants
                          if not variant.islower() and not variant.isupper()]
                cased = [variant for variant in variants
                         if not variant.islower()]
                canonical = (proper or cased or variants)[0]

                for variant in variants:
                    if variant != canonical:
                        mapping[variant] = canonical

        if mapping:
            count = int(frame[column].isin(mapping).sum())
            frame[column] = frame[column].replace(mapping)
            add_action(report, column, "merged_category_variants",
                       f"Merged spelling variants: {mapping}.", count)

    return frame


def flag_missing_values(frame, report):
    for column in frame.columns:
        missing = int(frame[column].isna().sum())

        if missing:
            add_flag(report, column, "missing_values",
                     f"{missing} of {len(frame)} values are missing "
                     f"({missing / len(frame):.0%}).")


def find_outliers(values):
    """
    Return the set of extreme values in a numeric Series.

    Rule 1: outside the IQR fences, only if such values are rare.
    Rule 2: the highest / lowest positive values when they are
            OUTLIER_GAP_FACTOR times away from the next value.
    """

    max_count = max(1, int(OUTLIER_MAX_SHARE * len(values)))
    outliers = set()

    # Rule 1: IQR fences.
    q1, q3 = values.quantile([0.25, 0.75])
    spread = OUTLIER_IQR_FACTOR * (q3 - q1)
    fenced = values[(values < q1 - spread) | (values > q3 + spread)]

    if len(fenced) <= max_count:
        outliers.update(fenced.tolist())

    # Rule 2: a big jump at either end of the sorted positive values.
    ordered = sorted(values[values > 0].unique())

    for position in range(len(ordered) - 1, 0, -1):
        if len(ordered) - position > max_count:
            break
        if ordered[position] >= OUTLIER_GAP_FACTOR * ordered[position - 1]:
            outliers.update(ordered[position:])
            break

    for position in range(min(max_count, len(ordered) - 1)):
        if ordered[position + 1] >= OUTLIER_GAP_FACTOR * ordered[position]:
            outliers.update(ordered[:position + 1])
            break

    return outliers


def flag_outliers(frame, report):
    for column, kind in report["column_types"].items():
        if kind not in {"integer", "float"}:
            continue

        values = frame[column].dropna().astype(float)

        if len(values) < 8:
            continue

        outliers = find_outliers(values)

        if outliers:
            count = int(values.isin(outliers).sum())
            examples = [
                int(value) if float(value).is_integer() else float(value)
                for value in sorted(outliers, key=abs, reverse=True)[:3]
            ]
            add_flag(report, column, "outliers",
                     f"{count} extreme value(s), e.g. {examples}; kept.")


def flag_similar_categories(frame, report):
    for column, kind in report["column_types"].items():
        if kind != "text" or not is_category_column(frame[column]):
            continue

        values = sorted(frame[column].dropna().unique())

        if len(values) > SIMILAR_CATEGORY_MAX_VALUES:
            continue

        pairs = []

        for position, first in enumerate(values):
            for second in values[position + 1:]:
                # "Product 1" vs "Product 2" differ only in digits.
                if re.sub(r"\d", "", first) == re.sub(r"\d", "", second):
                    continue

                ratio = difflib.SequenceMatcher(
                    None, first.casefold(), second.casefold()
                ).ratio()

                if ratio >= SIMILAR_CATEGORY_RATIO:
                    pairs.append((first, second))

        if pairs:
            add_flag(report, column, "similar_categories",
                     f"Possibly the same category: {pairs[:5]}; not merged.")


def flag_constant_columns(frame, report):
    for column in frame.columns:
        if len(frame) > 1 and frame[column].nunique(dropna=False) == 1:
            add_flag(report, column, "constant_column",
                     f"Every row has the same value "
                     f"({frame[column].iloc[0]!r}).")

## 8. Full cleaning for one file

`load_and_clean(file_path)` is the single entry point that the API will
call. It returns one result per table (one per Excel sheet).

In [8]:
# ============================================================
# 8. FULL CLEANING FOR ONE FILE
# ============================================================

def make_table_name(file_path, sheet, sheet_count):
    """sales.csv -> sales; book.xlsx sheet 'Q1' -> book_q1"""

    name = clean_column_name(Path(file_path).stem, 0)

    if sheet is not None and sheet_count > 1:
        name = f"{name}_{clean_column_name(sheet, 0)}"

    return name


def clean_table(table, table_name, source_file):
    """
    Clean one grid table.

    Returns:
        (cleaned DataFrame, raw DataFrame, report dict)
    """

    report = new_report(table_name, source_file)
    report["source_info"] = table["source_info"]

    # Step 1: header row and column names.
    headers, body, header_row = split_header(table)
    names, renamed = clean_column_names(headers)
    body.columns = names

    report["header_row"] = header_row
    report["renamed_columns"] = renamed
    report["rows_before"] = len(body)
    report["columns_before"] = len(names)

    if header_row:
        add_action(report, None, "skipped_title_rows",
                   f"Header found on row {header_row + 1}; rows above it "
                   "were dropped.", header_row)

    # Step 2: raw copy keeps the original values.
    raw = body.copy()

    # Step 3: value cleaning.
    frame = body.copy()
    frame = trim_whitespace(frame, report)
    frame = replace_null_placeholders(frame, report)
    frame = drop_empty_rows_and_columns(frame, report)
    frame = drop_duplicate_rows(frame, report)

    if frame.empty:
        raise ValueError(f"{table_name}: no data rows left after cleaning.")

    # Step 4: types, then categories.
    for column in frame.columns:
        frame = convert_column(frame, column, report)

    frame = merge_category_variants(frame, report)

    # Step 5: flags only.
    flag_missing_values(frame, report)
    flag_outliers(frame, report)
    flag_similar_categories(frame, report)
    flag_constant_columns(frame, report)

    report["rows_after"] = len(frame)
    report["columns_after"] = len(frame.columns)

    logger.info(
        "Cleaned %s | rows %d -> %d | columns %d -> %d | "
        "%d action(s), %d flag(s)",
        table_name, report["rows_before"], report["rows_after"],
        report["columns_before"], report["columns_after"],
        len(report["actions"]), len(report["flags"]),
    )

    return frame, raw, report


def load_and_clean(file_path):
    """
    Read a file and clean every table in it.

    Returns:
        list of {"table_name", "cleaned", "raw", "report"}
    """

    file_path = Path(file_path)
    tables = read_file(file_path)
    results = []

    for table in tables:
        table_name = make_table_name(file_path, table["sheet"], len(tables))
        cleaned, raw, report = clean_table(table, table_name, file_path.name)
        results.append({"table_name": table_name, "cleaned": cleaned,
                        "raw": raw, "report": report})

    return results


def print_report(report):
    """Readable summary of one cleaning report."""

    print(f"\n=== {report['table_name']} ({report['source_file']}) ===")
    print(f"rows {report['rows_before']} -> {report['rows_after']} | "
          f"columns {report['columns_before']} -> {report['columns_after']}")
    print("types:", report["column_types"])

    print("\nApplied:")
    for action in report["actions"]:
        column = f"[{action['column']}] " if action["column"] else ""
        print(f"  - {column}{action['detail']}")

    print("\nFlagged (unchanged):")
    for flag in report["flags"]:
        print(f"  - [{flag['column']}] {flag['issue']}: {flag['detail']}")

## 9. Run on the sample files

In [9]:
# ============================================================
# 9. RUN ON THE SAMPLE FILES
# ============================================================

all_results = {}

for sample in sample_files():
    if sample.suffix.lower() not in SUPPORTED_EXTENSIONS:
        continue

    for result in load_and_clean(sample):
        all_results[result["table_name"]] = result
        print_report(result["report"])

2026-11-14 19:25:50,662 | INFO | Read customers.json | records=8


2026-11-14 19:25:50,718 | INFO | Cleaned customers | rows 8 -> 8 | columns 6 -> 6 | 3 action(s), 0 flag(s)


2026-11-14 19:25:50,729 | INFO | Read messy_employees.xlsx | sheets=2



=== customers (customers.json) ===
rows 8 -> 8 | columns 6 -> 6
types: {'customer_id': 'integer', 'name': 'text', 'signup': 'date', 'tags': 'text', 'address_city': 'text', 'address_pin': 'integer'}

Applied:
  - [customer_id] Converted text to integer.
  - [signup] Converted text to date (format %Y-%m-%d).
  - [address_pin] Converted text to integer.

Flagged (unchanged):


2026-11-14 19:25:50,783 | INFO | Cleaned messy_employees_employees | rows 20 -> 20 | columns 7 -> 7 | 4 action(s), 2 flag(s)


2026-11-14 19:25:50,799 | INFO | Cleaned messy_employees_targets | rows 4 -> 4 | columns 3 -> 3 | 2 action(s), 0 flag(s)


... (2 outputs left out)


=== messy_employees_employees (messy_employees.xlsx) ===
rows 20 -> 20 | columns 7 -> 7
types: {'emp_id': 'text', 'full_name': 'text', 'department': 'text', 'joining_date': 'date', 'salary': 'text', 'last_review': 'text', 'active': 'boolean'}

Applied:
  - Header found on row 3; rows above it were dropped.
  - [joining_date] Converted text to date (format %Y-%m-%d).
  - [active] Converted text to boolean.
  - [department] Merged spelling variants: {'sales': 'Sales', 'hr': 'HR'}.

Flagged (unchanged):
  - [salary] partly_numeric: 90% of values are numbers; kept as text. Not numbers: ['confidential']
  - [last_review] ambiguous_dates: Dates fit both day-first and month-first (%d/%m/%Y / %m/%d/%Y); kept as text.

=== messy_employees_targets (messy_employees.xlsx) ===
rows 4 -> 4 | columns 3 -> 3
types: {'department': 'text', 'q1_target': 'integer', 'q2_target': 'integer'}

Applied:
  - [q1_target] Converted text to integer.
  - [q2_target] Converted text to integer.

Flagged (unchanged):

2026-11-14 19:25:50,931 | INFO | Read products.parquet | rows=5


2026-11-14 19:25:50,954 | INFO | Cleaned products | rows 5 -> 5 | columns 3 -> 3 | 1 action(s), 0 flag(s)



=== messy_sales (messy_sales.csv) ===
rows 35 -> 32 | columns 15 -> 14
types: {'order_id': 'text', 'order_date': 'date', 'customer_name': 'text', 'city': 'text', 'region': 'text', 'product': 'text', 'units_sold': 'integer', 'unit_price': 'integer', 'discount_pct': 'integer', 'total_sales': 'integer', 'paid': 'boolean', 'sales_rep': 'text', 'sales_rep_2': 'text', 'notes': 'text'}

Applied:
  - Header found on row 4; rows above it were dropped.
  - Removed extra spaces inside text values.
  - [units_sold] Treated ['-', 'N/A'] as missing.
  - [notes] Treated ['null'] as missing.
  - Removed empty columns ['column_15'].
  - Removed rows with no values.
  - Removed exact duplicate rows.
  - [order_date] Converted text to date (format %d/%m/%Y).
  - [units_sold] Converted text to integer; values that did not parse became null: ['twelve'].
  - [unit_price] Converted text to integer; removed currency symbols.
  - [discount_pct] Converted text to integer; removed percent symbols.
  - [total_sa

In [10]:
sales = all_results["messy_sales"]
print("RAW (first 5 rows)")
display(sales["raw"].head())
print("CLEANED (first 5 rows)")
display(sales["cleaned"].head())
print(sales["cleaned"].dtypes)

RAW (first 5 rows)


,order_id,order_date,customer_name,city,region,product,units_sold,unit_price,discount_pct,total_sales,paid,sales_rep,sales_rep_2,notes,column_15
0,00101,08/01/2024,Rahul Singh,Delhi,North,Keyboard,7,"$1,500.00",10%,"$9,450.00",yes,Kiran,Meera,None,None
1,00102,06/01/2024,Rahul Singh,Delhi,North,Headphones,9,"$3,500.00",0%,"$31,500.00",N,Meera,Kiran,None,None
2,00103,09/01/2024,Amit Shah,delhi,North,Monitor,9,"$12,000.00",5%,"$102,600.00",Yes,Kiran,Sanjay,Gift wrap,None
3,00104,21/02/2024,Asha Verma,Delhi,North,Headphones,N/A,"$3,500.00",10%,"$31,500.00",No,Kiran,Sanjay,None,None
4,00105,11/03/2024,Rahul Singh,Mumbai,West,Monitor,7,"$12,000.00",0%,"$84,000.00",yes,Meera,Sanjay,None,None


CLEANED (first 5 rows)


,order_id,order_date,customer_name,city,region,product,units_sold,unit_price,discount_pct,total_sales,paid,sales_rep,sales_rep_2,notes
0,00101,2024-01-08,Rahul Singh,Delhi,North,Keyboard,7,1500,10,9450,True,Kiran,Meera,NaN
1,00102,2024-01-06,Rahul Singh,Delhi,North,Headphones,9,3500,0,31500,False,Meera,Kiran,NaN
2,00103,2024-01-09,Amit Shah,Delhi,North,Monitor,9,12000,5,102600,True,Kiran,Sanjay,Gift wrap
3,00104,2024-02-21,Asha Verma,Delhi,North,Headphones,<NA>,3500,10,31500,False,Kiran,Sanjay,NaN
4,00105,2024-03-11,Rahul Singh,Mumbai,West,Monitor,7,12000,0,84000,True,Meera,Sanjay,NaN


order_id                    str
order_date       datetime64[us]
customer_name               str
city                        str
region                      str
product                     str
units_sold                Int64
unit_price                Int64
discount_pct              Int64
total_sales               Int64
paid                    boolean
sales_rep                   str
sales_rep_2                 str
notes                       str
dtype: object


## 10. Checks

Each planted problem in the sample files must come out as expected.
These asserts become unit tests after conversion.

In [11]:
# ============================================================
# 10. CHECKS
# ============================================================

def actions_of(report, action):
    return [item for item in report["actions"] if item["action"] == action]


def flags_of(report, issue):
    return [item["column"] for item in report["flags"]
            if item["issue"] == issue]


sales_df = sales["cleaned"]
sales_report = sales["report"]
types = sales_report["column_types"]

# Header and names
assert sales_report["header_row"] == 3
assert list(sales_df.columns) == [
    "order_id", "order_date", "customer_name", "city", "region", "product",
    "units_sold", "unit_price", "discount_pct", "total_sales", "paid",
    "sales_rep", "sales_rep_2", "notes",
]

# Rows: 32 orders + 2 duplicates + 1 empty row -> 32
assert sales_report["rows_before"] == 35
assert len(sales_df) == 32
assert actions_of(sales_report, "dropped_duplicate_rows")[0]["count"] == 2

# Types
assert types["order_id"] == "text"                       # leading zeros
assert "order_id" in flags_of(sales_report, "identifier_kept_as_text")
assert types["order_date"] == "date"
assert types["units_sold"] == "integer"                 # "twelve" -> null
assert types["unit_price"] == "integer"
assert types["total_sales"] == "integer"                # all whole numbers
assert types["discount_pct"] == "integer"
assert types["paid"] == "boolean"
assert sales_report["column_units"]["unit_price"] == "currency"
assert sales_report["column_units"]["discount_pct"] == "percent"

# Values
assert sales_df["units_sold"].isna().sum() == 3          # N/A, -, twelve
assert sales_df["customer_name"].str.contains("  ").sum() == 0
assert set(sales_df["city"]) <= {"Delhi", "Mumbai", "Mumbay",
                                 "Bengaluru", "Pune", "New Delhi"}

# Flags: only the planted 9,999,999 is an outlier, not every laptop sale
assert flags_of(sales_report, "outliers") == ["total_sales"]
assert "9999999" in [f for f in sales_report["flags"]
                     if f["issue"] == "outliers"][0]["detail"]
assert "city" in flags_of(sales_report, "similar_categories")
assert "notes" in flags_of(sales_report, "missing_values")

# Excel
employees = all_results["messy_employees_employees"]["report"]
assert employees["header_row"] == 2
assert "nan" not in all_results["messy_employees_employees"]["cleaned"].columns
assert employees["column_types"]["joining_date"] == "date"
assert employees["column_types"]["active"] == "boolean"
assert "salary" in flags_of(employees, "partly_numeric")
assert "last_review" in flags_of(employees, "ambiguous_dates")
assert "messy_employees_targets" in all_results

# JSON and Parquet
customers = all_results["customers"]["cleaned"]
assert {"address_city", "address_pin", "signup"} <= set(customers.columns)
assert all_results["customers"]["report"]["column_types"]["signup"] == "date"
assert all_results["products"]["report"]["column_types"]["list_price"] == "integer"

# Rule: one stray value in a small column still allows conversion.
assert type_matches(17, 18) and not type_matches(16, 18)
assert not type_matches(2, 3)                          # 67 % is too low
small = pd.DataFrame({"hours": [str(i) for i in range(17)] + ["unknown"]})
small_report = new_report("t", "t.csv")
small = convert_column(small, "hours", small_report)
assert small_report["column_types"]["hours"] == "integer"
assert small["hours"].isna().sum() == 1

# Rule: normal casing beats ALL CAPS; an acronym keeps its capitals.
merge = pd.DataFrame({"c": ["BEVERAGES"] * 3 + ["Beverages"] + ["beverages"]
                           + ["hr", "hr", "HR"] + ["x"] * 4})
merge_report = new_report("t", "t.csv")
merge_report["column_types"]["c"] = "text"
merge = merge_category_variants(merge, merge_report)
assert set(merge["c"]) == {"Beverages", "HR", "x"}

print("All checks passed.")

All checks passed.


## 11. Save outputs

Notebook 02 reads these: cleaned and raw tables as Parquet (keeps the
types) and the report as JSON.

In [12]:
# ============================================================
# 11. SAVE OUTPUTS
# ============================================================

def save_result(result, output_dir):
    """Write cleaned + raw Parquet and the report JSON for one table."""

    name = result["table_name"]

    result["cleaned"].to_parquet(output_dir / f"{name}.parquet", index=False)
    result["raw"].to_parquet(output_dir / f"{name}_raw.parquet", index=False)

    with (output_dir / f"{name}_report.json").open("w", encoding="utf-8") as f:
        json.dump(result["report"], f, indent=2, ensure_ascii=False,
                  default=str)


for result in all_results.values():
    save_result(result, OUTPUT_DIR)

print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

['customers.parquet', 'customers_raw.parquet', 'customers_report.json', 'messy_employees_employees.parquet', 'messy_employees_employees_raw.parquet', 'messy_employees_employees_report.json', 'messy_employees_targets.parquet', 'messy_employees_targets_raw.parquet', 'messy_employees_targets_report.json', 'messy_sales.parquet', 'messy_sales_raw.parquet', 'messy_sales_report.json', 'products.parquet', 'products_raw.parquet', 'products_report.json']
